# **Insurance Management System - CS6100 Project** 

By Sri Surya Sameer Vaddhiparthy and Dajana Muho


Under the guidance of Professor Dr. Ajay Gupta

# Data Analytics, Visualization, Model Computation and Predictions

## Imports

### Importing libraries

In [ ]:
import pandas as pd
import os
import seaborn as sns
import numpy as np
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import string
import random
import uuid
from collections import Counter
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import KMeansSMOTE
from imblearn.under_sampling import RandomUnderSampler

### Importing Data

In [ ]:
# Get the current working directory
cwd = os.getcwd()

# Print the current working directory
print("Current working directory: {0}".format(cwd))

# Loading the data
df = pd.read_csv (r'telematics_syn-032021.csv')

## Understanding the data

In [ ]:
df.columns

### Variable names and descriptions.

#### Traditionally gathered data

Duration - Duration of the insurance coverage of a given policy, in days

Insured.age - Age of insured driver, in years

Insured.sex - Sex of insured driver (Male/Female)

Car.age - Age of vehicle, in years

Marital - Marital status (Single/Married)

Car.use - Use of vehicle: Private, Commute, Farmer, Commercial

Credit.score - Credit score of insured driver

Region - Type of region where driver lives: rural, urban

Annual.miles.drive - Annual miles expected to be driven declared by driver

Years.noclaims - Number of years without any claims

Territory - Territorial location of vehicle


#### Telematically sensed data

Annual.pct.driven - Annualized percentage of time on the road

Total.miles.driven - Total distance driven in miles

Pct.drive.xxx - Percent of driving day xxx of the week: mon/tue/. . . /sun

Pct.drive.xhrs - Percent vehicle driven within x hrs: 2hrs/3hrs/4hrs

Pct.drive.xxx - Percent vehicle driven during xxx: wkday/wkend

Pct.drive.rushxx - Percent of driving during xx rush hours: am/pm

Avgdays.week - Mean number of days used per week

Accel.xxmiles - Number of sudden acceleration 6/8/9/. . . /14 mph/s per 1000 miles

Brake.xxmiles - Number of sudden brakes 6/8/9/. . . /14 mph/s per 1000 miles

Left.turn.intensityxx - Number of left turn per 1000 miles with intensity 08/09/10/11/12

Right.turn.intensityxx - Number of right turn per 1000 miles with intensity 08/09/10/11/12

Response NB_Claim - Number of claims during observation

AMT_Claim - Aggregated amount of claims during observation


Source of dataset description- Article: Synthetic Dataset Generation of Driver Telematics by Banghee So, Jean-Philippe Boucher and Emiliano A. Valdez

#### Summary Statistics

In [ ]:

df.describe().T

#### Missing Data

In [ ]:
df.isna().sum()

## Dataset customization

### Removing "." and spaces from column names

In [ ]:
# getting a list of column names
col_list=df.columns.values.tolist()

#Renaming the columns by replacing "." and with underscores
for i in range(0,len(col_list)):
    col_list[i]=col_list[i].replace(".", "_")

#Removing spaces in variable names
for i in range(0,len(col_list)):
    col_list[i]=col_list[i].replace(" ", "")
#reassigning the new list of column names  
df.columns = col_list

### Removing null values

In [ ]:
# Drop rows with null values (if any)
df.dropna(axis = 1)

#Verify number of null values in each column
df.isnull().sum(axis = 0)

### Assigning unique id to each row

In [ ]:
df['licensePlate']=df.index+1

original_ids = df['licensePlate'].unique()
DIGITS = 9  # number of hex digits of the UUID to use
new_ids = {cid: int(uuid.uuid4().hex[:DIGITS], base=16) for cid in original_ids}
df['licensePlate'] = df['licensePlate'].map(new_ids)

In [ ]:
df['licensePlate'] = df['licensePlate'].astype(str)

u=list(string.ascii_uppercase)
df['licensePlate']=(df['licensePlate'].str.replace('[1-2]',lambda x: random.choice(u)))

In [ ]:
df['licensePlate'].head()

In [ ]:
df['licensePlate'].nunique()

## Splitting data

### Selecting customer data with a non-zero claim amount.

In [ ]:

df_claim = df[df['AMT_Claim']!=0]
df_claim 

In [ ]:
df_claim.describe()

### Selecting customer data with a zero claim amount.

In [ ]:
#getting values with non-zero insurance claim
df_noclaim = df[df['AMT_Claim']==0]
df_noclaim 

In [ ]:
df_noclaim.describe()

### Visualization to understand the data 

#### Age Distribution

In [ ]:
df.hist(column='Car_age', grid=False, color='#A8A8A8')

In [ ]:
df.hist(column='Insured_age', grid=False, color='#A8A8A8')

In [ ]:
bins=np.arange(20, 100, 10).tolist()
labels = ['20s','30s','40s','50s','60s','70s','80s',]
df['AgeGroup'] = pd.cut(df['Insured_age'], bins=bins, labels=labels, right=False)




sns.countplot(data=df,y='AgeGroup')

In [ ]:
df[['Insured_age','AgeGroup']]

In [ ]:
#sns.countplot(data=df,y='AMT_Claim')
df_noclaim.hist(column='Annual_miles_drive', bins=10, grid=False, color='#A8A8A8')

In [ ]:
df_claim.hist(column='Annual_miles_drive', bins=10, grid=False, color='#A8A8A8')

In [ ]:
df_noclaim.hist(column='Years_noclaims', bins=10, grid=False, color='#A8A8A8')

In [ ]:
bins=np.arange(0, 11000, 1000).tolist()
labels = ['1s','2s','3s','4s','5s','6s','7s','8s','9s','10s']
df['ClaimGroup'] = pd.cut(df['AMT_Claim'], bins=bins, labels=labels, right=False)




In [ ]:
bins=np.arange(400, 1100, 100).tolist()
labels = ['400s','500s','600s','700s','800s','900s']
df['CSgroup'] = pd.cut(df['Credit_score'], bins=bins, labels=labels, right=False)




In [ ]:
df[['Credit_score','CSgroup']]

In [ ]:
df_noclaim.describe()

## Logistic Regression model development for Claim(boolean)

### Converting NB_Claim into a binary variable to use for predicting claim or no-claim

In [ ]:
df['NB_Claim'] = df['NB_Claim'].replace([2,3],1)

### Feature Engineering for Machine Learning model to predict Claim(boolean)

In [ ]:
df.columns

In [ ]:
##Using correlation to select features for NB_Claim
###Correlation with Outcome- Claim Amount
cor=df.corr()
cor_outcome = abs(cor["NB_Claim"])

In [ ]:
###Getting list of features with correlation <=0.02 with NB_Claim
low_corr_features = cor_outcome[cor_outcome<=0.02]                  
low_corr_features

In [ ]:
###Features with corr value above 0.2 with NB_Claim
abv02_corr_outcome = cor_outcome[cor_outcome>0.02]
abv02_corr_outcome

In [ ]:
#Selecting features with correlation above 0.2 with the final outcome NB_Claim (boolean)
X_df= df[["Duration","Insured_age","Car_age","Credit_score","Annual_miles_drive","Years_noclaims","Annual_pct_driven","Total_miles_driven","Pct_drive_2hrs","Pct_drive_rushpm","Avgdays_week","Accel_06miles","Brake_06miles","Brake_08miles"]]
y_df=df['NB_Claim']

In [ ]:
#Split the data into training set and test set. Use train test split() with test size = 0.5
from sklearn.model_selection import train_test_split
X1train, X1test, y1train, y1test = train_test_split(X_df, y_df, test_size = 0.2)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X1train = sc.fit_transform(X1train)
X1test = sc.transform(X1test)

### Logistic Regression

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


model3 = LogisticRegression(multi_class='ovr', random_state = 0)
model3.fit(X1train, y1train)
y1test = model3.predict(X1test)

# Model Evaluation
m3_train=model3.score(X1train, y1train)
m3_test=model3.score(X1test, y1test)

In [ ]:
print("For the model:", model3, ", the training accuracy is",m3_train,"and the testing accuracy is",m3_test)

### Random forest classsifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report


model4 = RandomForestClassifier(n_estimators = 50, max_depth = 10,min_samples_leaf=1, min_samples_split=2)
model4.fit(X1train, y1train)
y1test = model4.predict(X1test)

# Model Evaluation
m4_train=model4.score(X1train, y1train)
m4_test=model4.score(X1test, y1test)

In [ ]:
print("For the model:", model4, ", the training accuracy is",m4_train,"and the testing accuracy is",m4_test)

### Stochastic Gradient Descent Classifier (SGD-Classifier) - To minimize the cost function of the gradient descent

In [ ]:
from sklearn.linear_model import SGDClassifier

sgdc = SGDClassifier(max_iter=1000, tol=0.01)
print(sgdc)

xtrain, xtest, ytrain, ytest = train_test_split(X_df, y_df, test_size = 0.15)
sgdc.fit(xtrain, ytrain)

sgdc.fit(xtrain, ytrain)
score = sgdc.score(xtrain, ytrain)
print("Training score: ", score)

ypred = sgdc.predict(xtest)
cm = confusion_matrix(ytest, ypred)
print(cm)

cr = classification_report(ytest, ypred)
print(cr)

## The Challenge

In [ ]:
Counter(df['NB_Claim'])

### Introduction to the challenge

In [ ]:
df.hist(column='NB_Claim', grid=False, color='#A8A8A8')

In [ ]:
print("The number of rows with no claims is",len(df_noclaim), "and the number of rows with claim data is",len(df_claim), "and hence the claims data percentage is only",len(df_claim)*100/len(df),"percent.")

Some of the companys may experience low claim rate, as fact in support of this statement 
In 2019, 5.1 percent of insured homes had a claim, according to ISO
source: https://www.iii.org/fact-statistic/facts-statistics-homeowners-and-renters-insurance

### Solution to the challenge - SMOTE (Synthetic Minority Oversampling Technique)

In [ ]:
#Selecting features with correlation above 0.2 with the final outcome Claim (boolean)
X_df= df[["Duration","Insured_age","Car_age","Credit_score","Annual_miles_drive","Years_noclaims","Annual_pct_driven","Total_miles_driven","Pct_drive_2hrs","Pct_drive_rushpm","Avgdays_week","Accel_06miles","Brake_06miles","Brake_08miles"]]
y_df=df['NB_Claim']

In [ ]:
Counter(y_df)

In [ ]:
sample_technique=SMOTE()
X, y = sample_technique.fit_resample(X_df, y_df)
Counter(y)

### Stochastic Gradient Descent Classifier after generating samples with Synthetic Minority Oversampling Technique

In [ ]:
from sklearn.linear_model import SGDClassifier

sgdc = SGDClassifier(max_iter=1000, tol=0.01,loss='log')
print(sgdc)

xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size = 0.15)
sgdc.fit(xtrain, ytrain)

sgdc.fit(xtrain, ytrain)
score = sgdc.score(xtrain, ytrain)
print("The training score for SGDC-SMOTE is ", score)

ypred = sgdc.predict(xtest)
cm = confusion_matrix(ytest, ypred)
print(cm)

cr = classification_report(ytest, ypred)
print(cr)

In [ ]:
sample_technique=SMOTETomek()
X, y = sample_technique.fit_resample(X_df, y_df)
Counter(y)

### Stochastic Gradient Descent Classifier after generating samples with SMOTE-TOMEK Links Method

In [ ]:
from sklearn.linear_model import SGDClassifier

sgdc2 = SGDClassifier(max_iter=1000, tol=0.01)
print(sgdc2)

xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size = 0.15)
sgdc2.fit(xtrain, ytrain)

sgdc2.fit(xtrain, ytrain)
score = sgdc2.score(xtrain, ytrain)
print("The training score for SGDC-SMOTE-TOMEK is : ", score)

ypred = sgdc2.predict(xtest)
cm = confusion_matrix(ytest, ypred)
print(cm)

cr = classification_report(ytest, ypred)
print(cr)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold

xtrain, xtest, ytrain, ytest=train_test_split(X, y, test_size=0.15)

xgbc = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
print(xgbc)

xgbc.fit(xtrain, ytrain)

# - cross validataion
scores = cross_val_score(xgbc, xtrain, ytrain, cv=5)
print("Mean cross-validation score: %.2f" % scores.mean())

kfold = KFold(n_splits=10, shuffle=True)
kf_cv_scores = cross_val_score(xgbc, xtrain, ytrain, cv=kfold )
print("K-fold CV average score: %.2f" % kf_cv_scores.mean())

ypred = xgbc.predict(xtest)
cm = confusion_matrix(ytest,ypred)
print(cm)

## Predicting values for Claim(boolean)

In [ ]:
df["predictedClaimValue"]= sgdc.predict(X_df)

In [ ]:
df['predictedClaimValue'] = df['predictedClaimValue'].astype(bool)

In [ ]:
df[["predictedClaimValue","NB_Claim"]].tail()

## Importing the final dataset to csv for transfer to front end

In [ ]:
df.columns

In [ ]:
df_ui=df[[
        "licensePlate",        
        "Left_turn_intensity08",
        "Left_turn_intensity09",
        "Left_turn_intensity10",
        "Left_turn_intensity11",
        "Left_turn_intensity12",
        "Pct_drive_mon",
        "Pct_drive_tue",
        "Pct_drive_wed",
        "Pct_drive_thr",
        "Pct_drive_fri",
        "Pct_drive_sat",
        "Pct_drive_sun",
        "Right_turn_intensity08",
        "Right_turn_intensity09",
        "Right_turn_intensity10",
        "Right_turn_intensity11",
        "Right_turn_intensity12",
        "Accel_06miles",
        "Accel_08miles",
        "Accel_09miles",
        "Accel_11miles",
        "Accel_12miles",
        "Accel_14miles",
        "Brake_06miles",
        "Brake_08miles",
        "Brake_09miles",
        "Brake_11miles",
        "Brake_12miles",
        "Brake_14miles",
        "predictedClaimValue"]]

In [ ]:
df_ui.to_csv('telematics_ui.csv')

In [ ]:
selected=df_ui.iloc[5030:5050].to_dict(orient='index')

In [ ]:
to_json=list(selected.values())

In [ ]:
to_json


In [ ]:
import json
with open('data.json', 'w') as f:
    json.dump(to_json, f)

## Future Scope with aid of more real world data:

1. More data will lead to increased accuracy of the Machine Learning model in predicting the Claim (True/False) and also computing the potential claim amount.
2. The process of Usage based Auto-insurance quote generation can be made more ML-Centric and highly automated by integrating our system into the existing insurance quote computing mechanisms.
3. Make the usage based insurance system a selling point by generating personalized savings computation based on user data inputs.

## References for source of data, description of dataset variables

1. Article: Synthetic Dataset Generation of Driver Telematics by Banghee So, Jean-Philippe Boucher and Emiliano A. Valdez
https://arxiv.org/abs/2102.00252

2. Sample projects listed in the class website.
https://cs.wmich.edu/gupta/teaching/cs6100/6100BigDataF21web/CS%206100%20Project%20Guidelines.html

3. SMOTE-TOMEK Technique
https://towardsdatascience.com/imbalanced-classification-in-python-smote-tomek-links-method-6e48dfe69bbc

4.SMOTE Technique
https://machinelearningmastery.com/smote-oversampling-for-imbalanced-classification/

